# Подготовка признаков для анализа конверсии

Сравниваем заполнение пропусков, обработку аномалий и отбор признаков
на Online Shoppers Purchasing Intention. Цель – `Revenue`, факт покупки.

## Содержание

1. [Данные](#data)
2. [Обучение и тест](#split)
3. [Компоненты конвейера](#components)
4. [Заполнение пропусков](#imputation)
5. [Аномальные сессии](#outliers)
6. [Генерация и отбор признаков](#selection)
7. [Итоговая оценка](#evaluation)
8. [Сохранение модели](#saving)

In [ ]:
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import Markdown, display
from imblearn.pipeline import Pipeline as SamplingPipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted

warnings.filterwarnings("error", category=ConvergenceWarning)
SEED = 42
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_PATH = ROOT / "data/raw/online_shoppers_intention.csv"
DATA_PATH = ROOT / "data/processed/online_shoppers_incomplete.csv"
MODEL_PATH = ROOT / "models/session_feature_selection.pkl"
NUMERIC = [
    "Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
CATEGORICAL = ["Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Weekend"]
GENERATED = ["TotalPages", "TotalDuration", "ProductTimePerPage"]

<a id="data"></a>
## Данные

Создать пропуски: `python -m src.data.create_incomplete_sessions`.
В каждом из трёх столбцов длительности случайно скрыто 12% значений.
Это MCAR: выбор позиции не зависит от признаков и покупки. Seed – 42.

In [ ]:
raw = pd.read_csv(RAW_PATH)
data = pd.read_csv(DATA_PATH)
print("Строк и столбцов:", data.shape)
print("Полных повторов в исходных данных:", raw.duplicated().sum())
display(data.isna().sum().loc[lambda counts: counts > 0].rename("Пропуски"))
display(data.head())

<a id="split"></a>
## Обучение и тест

Разделяем группы одинаковых исходных сессий в пропорции 80/20 со стратификацией.
Тест используем только для итоговой оценки. Все сравнения проходят на пяти одинаковых фолдах.

In [ ]:
if not data.columns.equals(raw.columns) or len(data) != len(raw):
    raise ValueError("Нарушено соответствие исходного и подготовленного CSV.")
if not ((data == raw) | data.isna()).all().all() or not data.Revenue.equals(raw.Revenue):
    raise ValueError("Ожидались только пропуски в признаках.")
keys = pd.MultiIndex.from_frame(raw.drop(columns="Revenue"))
groups = pd.factorize(keys)[0]
y = data.Revenue.astype(int)
unique = pd.DataFrame({"group": groups, "target": y})
if unique.groupby("group").target.nunique().max() > 1:
    raise ValueError("Совпадающие признаки имеют разные цели.")
unique = unique.drop_duplicates("group")
train_groups, test_groups = train_test_split(
    unique.group, test_size=0.2, random_state=SEED, stratify=unique.target,
)
train_mask = np.isin(groups, train_groups)
X_train = data.loc[train_mask].drop(columns="Revenue")
X_test = data.loc[~train_mask].drop(columns="Revenue")
y_train, y_test = y[train_mask], y[~train_mask]
train_groups = groups[train_mask]
cv = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X_train, y_train, train_groups))
display(pd.DataFrame({
    "строк": [len(X_train), len(X_test)],
    "групп": [len(set(train_groups)), len(set(groups[~train_mask]))],
    "доля покупок": [y_train.mean(), y_test.mean()],
}, index=["train", "test"]))

<a id="components"></a>
## Компоненты конвейера

Числовые признаки масштабируются перед импутацией. Категории заполняются модой и кодируются one-hot.
Модель во всех сравнениях – логистическая регрессия с одинаковыми параметрами.

In [ ]:
class SessionFeatures(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        result = X.copy()
        result["TotalPages"] = X[["Administrative", "Informational", "ProductRelated"]].sum(axis=1)
        result["TotalDuration"] = X[["Administrative_Duration", "Informational_Duration", "ProductRelated_Duration"]].sum(axis=1, skipna=False)
        result["ProductTimePerPage"] = (
            X["ProductRelated_Duration"] / X["ProductRelated"].replace(0, np.nan)
        ).mask(X["ProductRelated"].eq(0), 0)
        return result


class TrainingOutlierFilter(BaseEstimator):
    def __init__(self, contamination=0.02, random_state=42):
        self.contamination = contamination
        self.random_state = random_state

    def fit_resample(self, X, y):
        self.detector_ = IsolationForest(
            n_estimators=100, contamination=self.contamination,
            random_state=self.random_state, n_jobs=1,
        )
        self.keep_ = self.detector_.fit_predict(X) == 1
        self.removed_ = int((~self.keep_).sum())
        return X[self.keep_], np.asarray(y)[self.keep_]


class ShapSelector(TransformerMixin, BaseEstimator):
    def __init__(self, fraction=0.6, sample_size=256, random_state=42):
        self.fraction = fraction
        self.sample_size = sample_size
        self.random_state = random_state

    def fit(self, X, y):
        forest = RandomForestClassifier(
            n_estimators=80, max_depth=8, min_samples_leaf=5,
            class_weight="balanced", random_state=self.random_state, n_jobs=1,
        ).fit(X, y)
        indices = np.random.default_rng(self.random_state).choice(
            len(X), min(self.sample_size, len(X)), replace=False,
        )
        values = shap.TreeExplainer(forest).shap_values(X[indices])
        if isinstance(values, list):
            values = values[1]
        elif values.ndim == 3:
            values = values[:, :, 1]
        self.importances_ = np.abs(values).mean(axis=0)
        count = max(1, int(np.ceil(X.shape[1] * self.fraction)))
        self.support_ = np.zeros(X.shape[1], dtype=bool)
        self.support_[np.argsort(-self.importances_, kind="stable")[:count]] = True
        return self

    def transform(self, X):
        check_is_fitted(self, "support_")
        return X[:, self.support_]

    def get_support(self):
        check_is_fitted(self, "support_")
        return self.support_

In [ ]:
def make_imputer(name):
    if name == "Median":
        return SimpleImputer(strategy="median")
    if name == "KNN":
        return KNNImputer(n_neighbors=5, weights="distance")
    if name == "Iterative":
        return IterativeImputer(max_iter=20, tol=0.01, random_state=SEED)
    raise ValueError(name)


def build_pipeline(imputation="Median", outliers=False, features=False, selection="none"):
    numeric = Pipeline([
        ("scaler", StandardScaler()),
        ("imputer", make_imputer(imputation)),
    ])
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric, NUMERIC + (GENERATED if features else [])),
        ("cat", categorical, CATEGORICAL),
    ])
    selectors = {"none": "passthrough", "VarianceThreshold": VarianceThreshold(0.01), "SHAP": ShapSelector()}
    return SamplingPipeline([
        ("features", SessionFeatures() if features else "passthrough"),
        ("preprocessor", preprocessor),
        ("outliers", TrainingOutlierFilter() if outliers else "passthrough"),
        ("selection", selectors[selection]),
        ("classifier", LogisticRegression(max_iter=3000, random_state=SEED)),
    ])


def compare(configurations):
    rows = []
    for name, settings in configurations.items():
        result = cross_validate(
            build_pipeline(**settings), X_train, y_train, cv=cv,
            scoring={"roc_auc": "roc_auc", "average_precision": "average_precision", "f1": "f1"},
            n_jobs=1, error_score="raise",
        )
        rows.append({
            "Вариант": name,
            "ROC-AUC": result["test_roc_auc"].mean(),
            "Разброс ROC-AUC": result["test_roc_auc"].std(),
            "Average Precision": result["test_average_precision"].mean(),
            "F1": result["test_f1"].mean(),
            "Секунд на fit": result["fit_time"].mean(),
        })
        print(name, "готово", flush=True)
    return pd.DataFrame(rows).set_index("Вариант")

<a id="imputation"></a>
## Заполнение пропусков

Сравниваем медиану, пять ближайших соседей и итеративное восстановление.
Импутеры и масштабирование обучаются заново на каждом фолде.

In [ ]:
imputation_table = compare({name: {"imputation": name} for name in ["Median", "KNN", "Iterative"]})
display(imputation_table.round(4))
best_imputation = imputation_table["ROC-AUC"].idxmax()
print("Лучший метод по ROC-AUC:", best_imputation)

Распределения сравниваем на скрытых значениях `ProductRelated_Duration` из train.
Оригинал используется только для графика. Шкала – секунды, по горизонтали symlog для длинного хвоста.

In [ ]:
column = "ProductRelated_Duration"
missing = X_train[column].isna()
reference = raw.loc[X_train.index[missing], column]
restored_values = {}
for name in imputation_table.index:
    numeric = build_pipeline(imputation=name).named_steps["preprocessor"].transformers[0][1]
    filled = numeric.fit_transform(X_train[NUMERIC])
    original_scale = numeric.named_steps["scaler"].inverse_transform(filled)
    restored_values[name] = original_scale[missing.to_numpy(), NUMERIC.index(column)]
all_values = np.concatenate([reference.to_numpy(), *restored_values.values()])
# Общие интервалы позволяют сравнить все методы на одной шкале.
low, high = all_values.min(), all_values.max()
bins = np.unique(np.r_[np.linspace(min(low, -1), 0, 12), 0, np.geomspace(1, max(high, 2), 35)])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
for ax, (name, values) in zip(axes, restored_values.items()):
    ax.hist(reference, bins=bins, alpha=0.45, label="До скрытия")
    ax.hist(values, bins=bins, alpha=0.45, label="После заполнения")
    ax.set(xscale="symlog", xlabel="Секунды", title=name)
    ax.legend()
axes[0].set_ylabel("Число значений")
plt.tight_layout()
plt.show()
display(pd.DataFrame({
    name: {"минимум": values.min(), "медиана": np.median(values), "максимум": values.max(), "отрицательных": int((values < 0).sum())}
    for name, values in restored_values.items()
}).T)

<a id="outliers"></a>
## Аномальные сессии

В данных есть длинные хвосты длительности и числа просмотров. Проверяем их на train.
IsolationForest выделяет 2% наиболее необычных сессий. Это не доказывает ошибку в данных.
Синтетические выбросы не добавляем: крайние наблюдения уже есть.

In [ ]:
tail_columns = ["ProductRelated", "Administrative_Duration", "Informational_Duration", "ProductRelated_Duration"]
q1, q3 = X_train[tail_columns].quantile(0.25), X_train[tail_columns].quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
display(pd.DataFrame({"Верхняя граница IQR": upper, "Максимум": X_train[tail_columns].max(), "Выше границы": X_train[tail_columns].gt(upper).sum()}))
diagnostic = build_pipeline(imputation=best_imputation, outliers=True).fit(X_train, y_train)
encoded = diagnostic.named_steps["preprocessor"].transform(X_train)
filter_step = diagnostic.named_steps["outliers"]
pca = PCA(n_components=2, random_state=SEED)
projection = pca.fit_transform(encoded)
fig, ax = plt.subplots(figsize=(8, 5))
for flag, label, color in [(True, "Остальные", "steelblue"), (False, "Аномальные", "tomato")]:
    mask = filter_step.keep_ == flag
    ax.scatter(projection[mask, 0], projection[mask, 1], s=10, alpha=0.5, c=color, label=label)
ax.set(xlabel="PC1", ylabel="PC2", title="IsolationForest на обучающей выборке")
ax.legend()
plt.tight_layout()
plt.show()
print("Доля дисперсии двух компонент:", pca.explained_variance_ratio_.sum().round(4))
display(pd.crosstab(pd.Series(np.asarray(y_train), name="Revenue"), pd.Series(~filter_step.keep_, name="Удалена")))

Удаление реализовано через `fit_resample` в imblearn Pipeline: строки и метки удаляются вместе.
При прогнозе фильтр пропускается, поэтому валидация и тест оцениваются целиком.

In [ ]:
stage_table = compare({
    "Удаление аномалий": {"imputation": best_imputation, "outliers": True},
    "Аномалии + новые признаки": {"imputation": best_imputation, "outliers": True, "features": True},
})
stage_table = pd.concat([imputation_table.loc[[best_imputation]].rename(index={best_imputation: "Только импутация"}), stage_table])
display(stage_table.round(4))

<a id="selection"></a>
## Генерация и отбор признаков

Добавлены общее число страниц, общая длительность и длительность просмотра на одну товарную страницу.
Нулевое число товарных страниц даёт ноль; пропущенная длительность остаётся пропуском до импутации.

VarianceThreshold убирает признаки с дисперсией ниже 0,01.
SHAP оставляет 60% признаков с наибольшим средним модулем вклада в прогноз RandomForest.
Лес и отбор обучаются только внутри train каждого фолда; для SHAP берём до 256 его строк.

In [ ]:
selection_table = compare({
    name: {"imputation": best_imputation, "outliers": True, "features": True, "selection": name}
    for name in ["VarianceThreshold", "SHAP"]
})
selection_table = pd.concat([
    stage_table.loc[["Аномалии + новые признаки"]].rename(index={"Аномалии + новые признаки": "Без отбора"}),
    selection_table,
])
display(selection_table.round(4))
best_selection = selection_table.loc[["VarianceThreshold", "SHAP"], "ROC-AUC"].idxmax()
print("Лучший метод отбора:", best_selection)
final_model = build_pipeline(
    imputation=best_imputation, outliers=True, features=True, selection=best_selection,
).fit(X_train, y_train)
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
selected = final_model.named_steps["selection"].get_support()
print("Признаков до отбора:", len(feature_names), "после:", int(selected.sum()))
display(pd.Series(feature_names[selected], name="Оставленные признаки"))

<a id="evaluation"></a>
## Итоговая оценка

Настройки выбраны по кросс-валидации. Итоговый конвейер включает генерацию признаков,
импутацию, удаление аномалий и отбор. Оцениваем его на всей отложенной выборке.

In [ ]:
prediction = final_model.predict(X_test)
probability = final_model.predict_proba(X_test)[:, 1]
test_metrics = pd.Series({
    "ROC-AUC": roc_auc_score(y_test, probability),
    "Average Precision": average_precision_score(y_test, probability),
    "Accuracy": accuracy_score(y_test, prediction),
    "Precision": precision_score(y_test, prediction, zero_division=0),
    "Recall": recall_score(y_test, prediction, zero_division=0),
    "F1": f1_score(y_test, prediction, zero_division=0),
}, name="Тест")
display(test_metrics.round(4))
display(pd.DataFrame(confusion_matrix(y_test, prediction), index=["Без покупки", "Покупка"], columns=["Прогноз: нет", "Прогноз: да"]))
print("Оценено строк:", len(prediction), "из", len(X_test))

In [ ]:
outlier_delta = stage_table.loc["Удаление аномалий", "ROC-AUC"] - stage_table.loc["Только импутация", "ROC-AUC"]
feature_delta = stage_table.loc["Аномалии + новые признаки", "ROC-AUC"] - stage_table.loc["Удаление аномалий", "ROC-AUC"]
selection_delta = selection_table.loc[best_selection, "ROC-AUC"] - selection_table.loc["Без отбора", "ROC-AUC"]
display(Markdown(
    f"Лучшее заполнение – **{best_imputation}**, отбор – **{best_selection}**. "
    f"Изменение среднего ROC-AUC: удаление аномалий **{outlier_delta:+.4f}**, "
    f"новые признаки **{feature_delta:+.4f}**, отбор **{selection_delta:+.4f}**. "
    f"На тесте ROC-AUC **{test_metrics['ROC-AUC']:.4f}**, F1 **{test_metrics['F1']:.4f}**."
))

По ROC-AUC методы заполнения почти равны: разница меньше 0,0002 при разбросе по фолдам около 0,0065.
Это не свидетельство устойчивого преимущества Iterative. Медиана быстрее и создаёт пик около 602 секунд.
KNN лучше сохраняет разброс. Iterative дал 15 отрицательных значений среди восстановленных
длительностей товарных просмотров в train. Они оставлены в эксперименте и отмечены как ограничение.

IsolationForest удалил 198 обучающих строк, в том числе 56 с покупкой.
На PCA часть аномалий лежит далеко от основной массы, часть смешана с ней:
две компоненты показывают только 36,3% дисперсии. Аномальность не означает ошибку записи.

Удаление аномалий и новые признаки немного снизили ROC-AUC. SHAP вернул часть качества
и сократил число признаков с 77 до 47, но итоговый результат кросс-валидации
не превзошёл конвейер с одной импутацией. Все стадии сохранены для сравнения подходов.

Выбор последовательный: сначала импутация, затем обработка и отбор.
Это не полный перебор сочетаний. Фолды использованы для выбора настроек,
поэтому окончательная оценка берётся с теста.

`PageValues` и итоговые показатели сессии могут быть недоступны до покупки.
Результаты относятся к завершённым сессиям, а случайное разделение не проверяет
перенос на будущие месяцы. MCAR не отражает все возможные причины пропусков.

<a id="saving"></a>
## Сохранение модели

Сохраняем полный конвейер через `pickle`. Для загрузки в новом ядре сначала
выполните импорты и определения `SessionFeatures`, `TrainingOutlierFilter`, `ShapSelector`.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_PATH.open("wb") as stream:
    pickle.dump(final_model, stream, protocol=pickle.HIGHEST_PROTOCOL)
print("Модель сохранена:", MODEL_PATH.relative_to(ROOT))